In [10]:
from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()


In [11]:
spark.sql("use spark_catalog")
spark.sql("use schema northwind_db")

DataFrame[]

In [12]:
df = spark.table("employees")

In [13]:
seed = 5
withReplacement = False
fraction = 0.1

df.sample(withReplacement, fraction, seed).show()


+----------+--------+---------+--------------------+---------------+--------------------+--------------------+--------------------+-------+------+----------+-------+--------------+---------+--------------------+---------+
|EmployeeID|LastName|FirstName|               Title|TitleOfCourtesy|           BirthDate|            HireDate|             Address|   City|Region|PostalCode|Country|     HomePhone|Extension|               Notes|ReportsTo|
+----------+--------+---------+--------------------+---------------+--------------------+--------------------+--------------------+-------+------+----------+-------+--------------+---------+--------------------+---------+
|         1| Davolio|    Nancy|Sales Representative|            Ms.|1948-12-08 00:00:...|1992-05-01 00:00:...|507 - 20th Ave. E...|Seattle|    WA|     98122|    USA|(206) 555-9857|     5467|"Education includ...|        2|
|         2|  Fuller|   Andrew|Vice President, S...|            Dr.|1952-02-19 00:00:...|1992-08-14 00:00:...|  

In [14]:
dfs = df.randomSplit([0.25, 0.75], seed)

print(f"rows in df inintial: {df.count()}")
print(f"rows in df 1: {dfs[0].count()}")
print(f"rows in df 2: {dfs[1].count()}")

df_joint = dfs[0].union(dfs[1])

print(f"rows in joint df: {df_joint.count()}")



rows in df inintial: 11
rows in df 1: 4
rows in df 2: 7
rows in joint df: 11


In [15]:
from pyspark.sql.functions import lit

df_range = spark.range(0,10000)
df_range = df_range.withColumn("string_col", lit("Just a string"))
display(df_range.limit(10))

DataFrame[id: bigint, string_col: string]

In [16]:
from pyspark.sql.functions import col

filter_expr = (col("id") >= 100) & (col("id") <= 200)
df_between_100_and_200 = df_range.filter(filter_expr).withColumn("filter_expr", filter_expr)
display(df_between_100_and_200)


DataFrame[id: bigint, string_col: string, filter_expr: boolean]

In [17]:
from pyspark.sql.functions import round, bround

df_round = df_range.select(round(lit(2.5)), bround(lit(2.5)))
display(df_round)

DataFrame[round(2.5, 0): double, bround(2.5, 0): double]